# 🧬 Introduction to Biopython

This notebook covers the fundamentals of working with biological sequences using **Biopython**.

**You will learn:**
1. Read FASTA files
2. Compute basic properties (length, GC content)
3. Translate DNA into protein
4. Query NCBI with Entrez
5. Run a basic BLAST search


## 1. Setup

Make sure Biopython is installed (`pip install biopython`).

In [ ]:
# Check Biopython is available
import Bio
print(f"Biopython version: {Bio.__version__}")


## 2. Working with sequences

Biopython's most basic object is `Seq`, which represents a biological sequence.


In [ ]:
from Bio.Seq import Seq

# Create a DNA sequence
dna = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")
print(f"Sequence:    {dna}")
print(f"Length:      {len(dna)} bp")
print(f"Complement:  {dna.complement()}")
print(f"Reverse complement: {dna.reverse_complement()}")
print(f"Transcription (DNA→RNA): {dna.transcribe()}")


## 3. GC content

GC content is a fundamental property: regions with high GC tend to be more thermally stable.

In [ ]:
from Bio.SeqUtils import gc_fraction

gc = gc_fraction(dna)
print(f"GC content: {gc:.2%}")

# Manually for sanity check
manual = (dna.count('G') + dna.count('C')) / len(dna)
print(f"Manual GC:  {manual:.2%}")


## 4. DNA → protein translation

The genetic code translates triplets of nucleotides (codons) to amino acids.
The default Biopython table is the standard genetic code (table 1).


In [ ]:
# Translate the DNA sequence to protein
protein = dna.translate()
print(f"DNA:     {dna}")
print(f"Protein: {protein}")

# The '*' character indicates a stop codon.
# We can stop translation at the first stop codon:
protein_to_stop = dna.translate(to_stop=True)
print(f"Up to stop: {protein_to_stop}")


## 5. Reading FASTA files

Real genomic data is delivered as FASTA. Let's create a small example file and parse it.


In [ ]:
import os
os.makedirs("../data", exist_ok=True)

# Tiny synthetic FASTA file for demonstration
fasta_content = '''>seq1 example sequence 1
ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
>seq2 example sequence 2
ATGAAACGCATTAGCACCACCATTACCACCACCATCACC
ATTACCACAGGTAACGGTGCGGGCTGA
>seq3 example sequence 3
ATGGGCCAGCAGCTGCAGCAGCAGCAGCAGCAGCAG
'''

with open("../data/example.fasta", "w") as f:
    f.write(fasta_content)

print("File created.")


In [ ]:
from Bio import SeqIO

# Iterate over records in a FASTA file
for record in SeqIO.parse("../data/example.fasta", "fasta"):
    print(f"ID: {record.id}")
    print(f"Description: {record.description}")
    print(f"Sequence: {record.seq}")
    print(f"Length: {len(record.seq)} bp")
    print(f"GC: {gc_fraction(record.seq):.2%}")
    print("-" * 50)


## 6. Querying NCBI with Entrez

Biopython lets us query NCBI databases programmatically. **Important**: NCBI requires an email so they can contact you in case of API misuse.

> **Note**: this cell requires an internet connection. If you don't have one, you can skip it.


In [ ]:
from Bio import Entrez

# REQUIRED: configure your email
Entrez.email = "your_email@example.com"  # ← change this

# Example: search for human insulin
try:
    handle = Entrez.esearch(db="nucleotide", term="insulin[Gene] AND human[Organism]", retmax=5)
    record = Entrez.read(handle)
    handle.close()
    print(f"Total results: {record['Count']}")
    print(f"First IDs: {record['IdList']}")
except Exception as e:
    print(f"Error (need internet): {e}")


In [ ]:
# Download a specific sequence (example: CYP2D6)
# Requires internet
try:
    handle = Entrez.efetch(db="nucleotide", id="NM_000106", rettype="fasta", retmode="text")
    cyp2d6 = SeqIO.read(handle, "fasta")
    handle.close()
    print(f"ID: {cyp2d6.id}")
    print(f"Length: {len(cyp2d6.seq)} bp")
    print(f"GC: {gc_fraction(cyp2d6.seq):.2%}")
    print(f"First 100 bases: {cyp2d6.seq[:100]}")
except Exception as e:
    print(f"Error (need internet): {e}")


## 7. Basic BLAST

BLAST (Basic Local Alignment Search Tool) finds similar sequences in massive databases. Biopython can run web BLAST, but it is slow (typically 1–5 minutes per query).

> **Note**: this cell takes a few minutes to run. It is shown here as a reference.


In [ ]:
# Example: BLAST a small sequence against the nr database
# UNCOMMENT to run (takes minutes)

# from Bio.Blast import NCBIWWW, NCBIXML
#
# query = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"
# result_handle = NCBIWWW.qblast("blastn", "nt", query)
#
# blast_record = NCBIXML.read(result_handle)
# for alignment in blast_record.alignments[:3]:
#     print(f"\n>>> {alignment.title}")
#     print(f"    Length: {alignment.length}")
#     for hsp in alignment.hsps[:1]:
#         print(f"    E-value: {hsp.expect}")
#         print(f"    Score: {hsp.score}")

print("BLAST commented out. Uncomment to run a real query.")


## 📝 Exercises

1. **Easy**: download the CYP2D6 sequence from NCBI and compute its GC content.
2. **Medium**: write a function that takes a DNA sequence and returns its 6 reading frames (3 forward + 3 reverse complement).
3. **Medium**: identify the longest ORF (Open Reading Frame) in each reading frame.
4. **Hard**: download CYP2D6, CYP2C9 and CYP3A4 from NCBI, save them in a multi-FASTA file and compute pairwise distances.
5. **Pharma-focused**: search for "warfarin metabolism" in PubMed via Entrez and list the 10 most recent papers.

## 📚 Resources

- [Biopython Tutorial](http://biopython.org/DIST/docs/tutorial/Tutorial.html)
- [NCBI Entrez Help](https://www.ncbi.nlm.nih.gov/books/NBK25497/)
- [Rosalind](https://rosalind.info/) — exercises that pair perfectly with this module
